# The perceptron in scikit-learn
100 students: CGPA, resume score and whether they were placed.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.linear_model import Perceptron
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# load the data and look at it
df = pd.read_csv("data/placement.csv")
print(df.shape)
df.head()

In [ ]:
# scatter plot: placed students in green, the others in red
px.scatter(df, x="cgpa", y="resume_score", color=df["placed"].map({1: "placed", 0: "not placed"}),
           color_discrete_map={"placed": "#54A24B", "not placed": "#E45756"}, template="simple_white")

## Training: the perceptron learns w1, w2 and b

In [ ]:
X = df[["cgpa", "resume_score"]]
y = df["placed"]
# random_state fixes the order in which the rows are visited
p = Perceptron(random_state=0)
p.fit(X, y)
print("w1, w2 =", p.coef_[0], " b =", p.intercept_[0])
print("training accuracy:", p.score(X, y))

## Prediction by hand: z = w1 x1 + w2 x2 + b, then the step function

In [ ]:
w1, w2 = p.coef_[0]
b = p.intercept_[0]
student = {"cgpa": 8.0, "resume_score": 7.0}
z = w1 * student["cgpa"] + w2 * student["resume_score"] + b
print("z =", round(z, 2), "-> prediction", 1 if z >= 0 else 0)
# scikit-learn gives the same answer
print("predict:", p.predict(pd.DataFrame([student]))[0])

## Decision regions (Plotly, in place of mlxtend's plot_decision_regions)

In [ ]:
# colour a fine grid of points by the class the model predicts there
def regions(model, title):
    gx, gy = np.meshgrid(np.linspace(5, 9.6, 300), np.linspace(4.8, 9.3, 300))
    grid = pd.DataFrame({"cgpa": gx.ravel(), "resume_score": gy.ravel()})
    zz = model.predict(grid).reshape(gx.shape)
    fig = go.Figure(go.Contour(x=gx[0], y=gy[:, 0], z=zz, showscale=False, opacity=0.25,
                               colorscale=[[0, "#E45756"], [1, "#54A24B"]], contours=dict(start=0.5, end=0.5, size=1)))
    for label, colour in [(1, "#54A24B"), (0, "#E45756")]:
        m = y == label
        fig.add_trace(go.Scatter(x=X.cgpa[m], y=X.resume_score[m], mode="markers", marker=dict(color=colour),
                                 name="placed" if label else "not placed"))
    fig.update_layout(template="simple_white", title=title, xaxis_title="CGPA", yaxis_title="Resume score")
    return fig

regions(p, f"Raw inputs: accuracy {p.score(X, y):.0%}")

## The same perceptron on standardized inputs

In [ ]:
# StandardScaler puts both columns on mean 0, standard deviation 1 before the perceptron sees them
ps = make_pipeline(StandardScaler(), Perceptron(random_state=0)).fit(X, y)
print("weights on scaled inputs:", ps[-1].coef_[0], " bias:", ps[-1].intercept_[0])
regions(ps, f"Standardized inputs: accuracy {ps.score(X, y):.0%}")